# GLM Lightning Distribution Analysis: May 2018 Case

```{image} ../thumbnails/GOES.png
:alt: Project Pythia logo
:width: 200px
```

# Overview

## This notebook will highlight the process of importing lightning data curtesy of the Geostationary Lightning Mapper (https://www.earthdata.nasa.gov/data/instruments/glm) from NASA's GOES (Geostationary Orbiting Environmental Satellite) 16 archive for an EF1 tornado event in New Hampshire in May of 2018. The GOES satellite has been providing continuous high resolution lightning data for the United States since 2017, and events since that time are available to access through the archive. Terrain basemaps will be imported from OpenTopoMap to examine potential effects in storm intensification over varying topography via lightning visualization proxy. 

### 1. Read in GLM data from NASA's GOES 16 archive

### 2. Limit the data to the spatial and temporal extent of interest

### 2. Define and import a terrain basemap

### 2. Generate a static lightning map for the hours leading up to the event

### 2. Generate an animated lightning map for the hours leading up to the event

# Prerequisites                        

| Concepts | Importance | Notes |
| --- | --- | --- |
| Concept                                  | Importance | Notes                                                                |
| Basic Python (lists, functions, loops)   | Necessary  | Notebook uses custom functions, loops, and basic data structures     |
| Pandas fundamentals                      | Helpful    | Filtering lightning data by time, combining DataFrames               |
| Intro to Xarray                          | Necessary  | GLM data are NetCDF files opened as xarray datasets                  |
| Understanding of NetCDF                  | Helpful    | Helps when interpreting variables, dimensions, metadata              |
| Remote data access (AWS S3 basics)       | Helpful    | GLM data come from the NOAA public S3 bucket                         |
| Intro to Cartopy                         | Necessary  | Required for projections, map display, coordinate transforms         |
| Plate Carrée vs Web Mercator projections | Necessary  | Understanding projections is essential for aligning layers correctly |
| Matplotlib plotting                      | Necessary  | Used for scatter and image overlays                                  |
| Requests + PIL                           | Helpful    | Used for downloading and stitching shaded-relief map tiles           |
| Latitude/Longitude concepts              | Necessary  | GLM flash data are in geographic coordinates                         |
| Timezones & UTC awareness                | Helpful    | GLM datasets use UTC timestamps                                      |


- **Time to learn**: 30 minutes.
- **System requirements**:
    - numpy  
    - pandas  
    - matplotlib  
    - cartopy  
    - xarray  
    - s3fs  
    - requests  
    - Pillow


# Import Packages

In [ ]:
import xarray as xr
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import pandas as pd
from datetime import datetime as dt, timezone, timedelta

import s3fs
import warnings
import math
import requests
from PIL import Image
from io import BytesIO

In [ ]:
fs = s3fs.S3FileSystem(anon=True)

In [ ]:
warnings.filterwarnings("ignore") 

# Define and Load in GLM data from NOAA GOES-16 archive set for desired datetime, region, across all flash variables.

In [ ]:
def list_glm_lcfa_files(dt, satellite="noaa-goes16"):
    year = dt.year
    doy = dt.timetuple().tm_yday
    hour = dt.strftime("%H")
    path = f"{satellite}/GLM-L2-LCFA/{year}/{doy:03d}/{hour}/"
    try:
        return fs.ls(path)
    except FileNotFoundError:
        return []

In [ ]:
def load_glm_hour(dt):
    files = list_glm_lcfa_files(dt)
    dfs = []

    for f in files:
        ds = xr.open_dataset(fs.open(f))

        # Extract flash-level variables
        df = ds[['flash_lat', 'flash_lon', 'flash_area', 'flash_energy']].to_dataframe()

        # Add correct timestamp
        df['time'] = ds['product_time'].values

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [ ]:
def load_glm_hour_filtered(dt, lat_min=42, lat_max=45, lon_min=-74, lon_max=-70):
    files = list_glm_lcfa_files(dt)
    dfs = []

    for f in files:
        ds = xr.open_dataset(fs.open(f))

        # Extract flash-level variables
        df = ds[['flash_lat', 'flash_lon', 'flash_area', 'flash_energy']].to_dataframe()

        # Add timestamp (use product_time)
        df['time'] = ds['product_time'].values

        # Apply spatial filter
        mask = (
            (df['flash_lat'] >= lat_min) &
            (df['flash_lat'] <= lat_max) &
            (df['flash_lon'] >= lon_min) &
            (df['flash_lon'] <= lon_max)
        )

        df = df[mask]

        if not df.empty:
            dfs.append(df)

    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        return pd.DataFrame()  # empty if no flashes

# Prepare GLM data to fit over terrain tile map.

In [ ]:
def deg2num(lat_deg, lon_deg, zoom):
    lat_rad = math.radians(lat_deg)
    n = 2.0 ** zoom
    xtile = int((lon_deg + 180.0) / 360.0 * n)
    ytile = int((1.0 - math.log(math.tan(lat_rad) + 1/math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return xtile, ytile

# Define time range of event (2 hours). This will take several minutes to complete.

In [ ]:
dfs = []
for hour in range(0, 2):   # loads 00 & 01 z
    t = dt(2018, 5, 5, hour)
    dfh = load_glm_hour_filtered(t)
    print(hour, len(dfh))
    dfs.append(dfh)

df_filtered = pd.concat(dfs, ignore_index=True)

# Define and import terrain basemap from OpenTopoMap. Generate static lightning map for 2 hours leading up to the event. Start location of the tornado indicated with upside down blue triangle.

In [ ]:
# inputs
lon_min, lon_max = -74, -70
lat_min, lat_max =  42,  45
zoom = 8 

# ---- 1) Compute tile index ranges with FLOOR ----
x_min, y_max = deg2num(lat_min, lon_min, zoom)  # SW corner
x_max, y_min = deg2num(lat_max, lon_max, zoom)  # NE corner

xs = list(range(min(x_min, x_max), max(x_min, x_max) + 1))
ys = list(range(min(y_min, y_max), max(y_min, y_max) + 1))

# Get Tiles
tiles = {}
for x in xs:
    for y in ys:
        url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        tiles[(x, y)] = Image.open(BytesIO(r.content))

# ---- 3) Stitch (Y increases south; first Y should go at the TOP) ----
# PIL image origin is upper-left, so the top row must be y_min.
tile_w, tile_h = next(iter(tiles.values())).size
W = len(xs) * tile_w
H = len(ys) * tile_h
background = Image.new("RGB", (W, H))

for x in xs:
    for y in ys:
        dx = (x - xs[0]) * tile_w
        dy = (y - ys[0]) * tile_h      # <-- keep this (no vertical flip!)
        background.paste(tiles[(x, y)], (dx, dy))

# ---- 4) Compute exact geographic extent of the stitched image (tile edges) ----
n = 2.0 ** zoom
def tile_x_to_lon(x):  return x / n * 360.0 - 180.0
def tile_y_to_lat(y):  # inverse WebMercator
    return math.degrees(math.atan(math.sinh(math.pi * (1 - 2*y/n))))

extent_tiles = [
    tile_x_to_lon(xs[0]),           # left
    tile_x_to_lon(xs[-1] + 1),      # right  (edge of the last tile)
    tile_y_to_lat(ys[-1] + 1),      # bottom (south edge)
    tile_y_to_lat(ys[0])            # top    (north edge)
]

# ---- 5) Plot (image in PlateCarree; lightning in PlateCarree) ----
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(projection=ccrs.PlateCarree()))
ax.imshow(background, extent=extent_tiles, origin="upper", transform=ccrs.PlateCarree())

# OPTIONAL: clip visible window to your requested bbox
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

# lightning
ax.scatter(df_filtered.flash_lon, df_filtered.flash_lat, s=3, color="red", transform=ccrs.PlateCarree())

# Add triangle (tornado) marker at 43.15 N, -72.45 W
ax.scatter(
    -72.45, 43.15,
    s=140,
    marker="v",           # upside-down triangle
    color="blue",
    edgecolor="black",
    linewidth=0.8,
    transform=ccrs.PlateCarree(),
    zorder=10
)

plt.title("05/04/18-05/05/18 Lightning Strikes with Shaded Relief")
plt.show()

# Define time range for animated lightning map (3 hours).

In [ ]:
dfs = []
times = []

for hour in range(0, 3):
    t = dt(2018, 5, 5, hour)
    dfh = load_glm_hour_filtered(t)
    print(hour, len(dfh))

    if not dfh.empty:
        dfs.append(dfh)
        times.append(t)

# Plot animated map of lightning for the 3 hours leading up to the event.

In [ ]:
# Build background map as before
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(projection=ccrs.PlateCarree()))
ax.imshow(background, extent=extent_tiles, origin="upper", transform=ccrs.PlateCarree())
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

# create an empty scatter object for animation
scatter = ax.scatter([], [], s=3, color="red", transform=ccrs.PlateCarree())

timestamp_text = ax.text(
    0.02, 0.97, "",
    transform=ax.transAxes,
    fontsize=16,
    color="black",
    weight="bold",
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        boxstyle="round,pad=0.3",
        alpha=0.7   # transparency
    ),
    zorder=20
)

def update_frame(i):
    df = dfs[i]
    scatter.set_offsets(np.column_stack([df.flash_lon, df.flash_lat]))
    timestamp_text.set_text(times[i].strftime("%Y-%m-%d %HZ"))
    return scatter, timestamp_text
    
from matplotlib.animation import FuncAnimation

anim = FuncAnimation(
    fig, update_frame, frames=len(dfs),
    interval=800, blit=True
)

plt.close()   # prevents duplicate display in Jupyter

from IPython.display import HTML
HTML(anim.to_jshtml())

# Summary
## Lightning activity ceased leading up to the Merrimack Tornado in the overnight hours with the loss of daytime heating. This lowered instability was likely compensated for with high shear and SRH, as seen in our earlier notebooks, and southerly low-level flow aided by the orientation of the Connecticut River Valley. While not directly related to tornadogenesis, lightning distribution can give important clues for the growth and development of supercell thunderstorms. It can be especially useful when plotted over terrain maps as an indirect way of visualizing changes related to wind fields and thermodynamic profiles over variable topography.


# What's Next?

## We will investigate another case, a stronger tornado in a more unstable mid-summer environment, to continue our analysis of lightning and its possible terrain influence.

## Resources and references

https://journals.ametsoc.org/view/journals/wefo/36/6/WAF-D-21-0018.1.xml

https://journals.ametsoc.org/view/journals/wefo/17/6/1520-0434_2002_017_1277_tiotot_2_0_co_2.xml